# Mission 3 ver18 — 전체 문맥 + 발화별 오심 근거 비교

**이 노트북 하나와 기존 `mission3_ver5_inputs.zip`으로 실행합니다.**
Colab에서 GPU 런타임을 선택하고 **런타임 → 모두 실행**을 누르세요.
Drive 연결 요청에는 입력 ZIP이 있는 **공용 계정**을 선택합니다.

ver16·17에서는 오심이 개선되지 않았습니다. 이번에는 긴 대화 안의 짧은 증상 표현과
앞뒤 질문·답변을 따로 살피면서 전체 대화도 유지하는 구조를 비교합니다.
새 방법의 성능과 GPU 실행 시간은 아직 측정하지 않았습니다. 큰 폭의 상승을 보장하지 않습니다.
원래 정답과 모든 샘플을 유지하며, 모델 입력은 `utterances[].text`뿐입니다.
threshold 탐색·사후 보정·외부 학습 데이터 추가 없이 증상별 두 class의 `argmax`로 예측합니다.

**이 노트북 한 번의 작업 예산은 설치·데이터 준비·학습·최종 평가를 포함해 180분입니다.**
Drive 로그인 대기 시간은 제외합니다. 느린 GPU에서는 추가 후보 또는 전체 Training 재학습이
생략될 수 있습니다. 완료된 모델만 최종 결과에 사용하며, 실제 학습 범위를 결과에 명시합니다.
작업 제한 시간과 모든 학습 단계의 완료는 다릅니다. 최종 완료 표시가 없는 실행은 미완료입니다.
결과 표시 후, 오류 발생 또는 시간 제한에 도달하면 GPU 런타임의 자동 반환을 요청합니다.
결과는 Drive에 남지만 `/content`의 임시 파일은 사라질 수 있습니다.

**이번에는 ver18 하나만 실행하면 됩니다. 이전 버전의 결과 파일은 필요하지 않습니다.**


## 이번에 비교하는 두 후보

- `baseline_ce`: ver15의 기본 분류 구조와 동일한 기준 모델.
- `utterance_evidence`: 같은 전체 문맥 분류기에 발화별 오심 근거를 읽는 작은 층을 추가.

각 대화 조각을 인코더에 추가로 반복 입력하지 않고, 기존 인코더의 결과를 두 경로에서 공유합니다.
새 경로는 각 발화와 앞뒤 발화를 순서대로 묶어 오심의 근거를 학습합니다.
긴 발화가 여러 조각으로 나뉘면 다시 같은 발화로 합치며, 겹친 토큰의 중복 기여를 줄입니다.
모든 발화를 사용하고 전체 문맥 경로도 유지합니다. 특정 단어가 없는 대화를 버리거나
구토·어지러움을 오심으로 바꾸는 규칙, 정답 수정, 발화자·의료 태그 입력은 사용하지 않습니다.
새 층은 오심 출력에만 학습 가능한 보정량을 더하며, 학습 후 점수 보정은 하지 않습니다.
공유 인코더도 함께 학습되므로 다른 증상 점수는 달라질 수 있습니다.

두 후보는 같은 공개 초기 모델, 분할·seed·학습 가중치를 사용합니다.
새 층의 초기 출력은 0이며, 처음에는 기본 모델과 같은 예측에서 시작합니다.
발화 단위 근거가 도움이 되는지 확인하는 실험이며, 부정·질문 문맥을 반드시 해결한다고 가정하지 않습니다.

두 후보의 양성 학습 가중치는 ver15의 `recall_moderate`와 같습니다.
`[고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란]` 순서로
`[1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]`을 사용합니다.
후보 학습은 최대 3epoch이며, 학습률 감소 길이는 기존과 같은 4epoch 기준을 유지합니다.
Training 내부 Macro F1으로 후보와 epoch를 선택하고, 시간이 허용되면 전체 Training으로 새로 학습합니다.
공식 Validation은 최종 모델을 정한 뒤 한 번 평가합니다. 개별 증상의 정밀도·재현율·F1도 저장합니다.
`candidate_comparison.json`에는 내부 평가의 증상별 변화와 기존 오답을 고친 수·새로 틀린 수를 저장합니다.
손실 함수는 기존 가중 cross entropy를 유지합니다. GCE 등 손실 변경은 이번 실험에 섞지 않습니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver18', 'backend': 'neural', 'model_id': 'klue/roberta-base', 'revision': '02f94ba5e3fcb7e2a58a390b8639b0fac974a8da', 'label_attention': False, 'contrastive_weight': 0.0, 'ce_views': 1, 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver18', 'work_dir': '/content/mission3_ver18', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'max_length': 512, 'overlap': 64, 'batch_size': 4, 'encoder_window_batch': 4, 'gradient_accumulation': 4, 'max_epochs': 3, 'scheduler_horizon_epochs': 4, 'patience': 2, 'learning_rate': 2e-05, 'head_learning_rate': 0.0001, 'refit_full_training': True, 'evidence_hidden': 128, 'evidence_dropout': 0.1, 'allow_cpu': False, 'log_every': 250, 'time_budget_minutes': 180, 'finalization_reserve_seconds': 1200, 'budget_safety_factor': 1.25, 'resume_run': ''}
AUTO_RELEASE_GPU = True # @param {type:"boolean"}
# Keep train_dir and validation_dir empty to use the existing input ZIP.
# Resume only a compatible run; set resume_run to its complete Drive directory.
# Each explicit rerun starts a NEW time budget and uses additional Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 3시간 작업 예산 시작 (이후 패키지 설치 시간도 포함)
import time
import threading
BUDGET_STARTED_MONOTONIC = time.monotonic()
DEADLINE_MONOTONIC = BUDGET_STARTED_MONOTONIC + float(SETTINGS["time_budget_minutes"]) * 60
SETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC
previous_timer = globals().get("RELEASE_TIMER")
if previous_timer is not None:
    previous_timer.cancel()
RELEASE_TIMER = None
RELEASE_LOCK = threading.Lock()
RELEASE_REQUESTED = False

def release_gpu():
    global RELEASE_REQUESTED
    if not AUTO_RELEASE_GPU:
        return
    with RELEASE_LOCK:
        if RELEASE_REQUESTED:
            return
        RELEASE_REQUESTED = True
        if RELEASE_TIMER is not None:
            RELEASE_TIMER.cancel()
    try:
        from google.colab import runtime as colab_runtime
        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)
        colab_runtime.unassign()
    except Exception as error:
        print("자동 반환 요청 실패:", type(error).__name__,
              "· Colab 메뉴 '런타임 → 연결 해제 및 런타임 삭제'를 직접 실행하세요.", flush=True)

if AUTO_RELEASE_GPU:
    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)
    RELEASE_TIMER.daemon = True
    RELEASE_TIMER.start()
print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")


In [ ]:
# @title 실행 코드 준비 (추가 파일 업로드 불필요)
import importlib.util
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Standalone Mission 3 categorical experiments embedded in ver10-ver13."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'neural.py': '"""Text-only dialogue classifiers with ordinary binary class decisions.\n\nEach symptom has two class logits. Predictions use the winning class directly;\nthere is no probability cutoff search, calibration, or post-hoc label weighting.\nPublic pretrained assets are prepared separately from local checkpoint weights.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom bisect import bisect_right\nfrom collections import Counter\nfrom numbers import Real\nfrom pathlib import Path\nimport random\nimport re\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nNUM_LABELS = 9\nCONTRASTIVE_DIM = 64\n_MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")\n\n\ndef seed_everything(seed):\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef prepare_encoder(model_id, revision, asset_dir):\n    """Download an encoder once; store only its config and tokenizer assets.\n\n    The caller stores the trained encoder and head together in its checkpoint.\n    Transformers imports are lazy so CPU tests do not require that dependency.\n    """\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n\n    if not isinstance(model_id, str) or not model_id.strip():\n        raise ValueError("model_id must be a nonempty string")\n    if not isinstance(revision, str) or not revision.strip():\n        raise ValueError("revision must be explicit")\n    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)\n    resolved = getattr(config, "_commit_hash", None)\n    if resolved is None and re.fullmatch(r"[0-9a-f]{40}", revision):\n        resolved = revision\n    if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-f]{40}", resolved):\n        raise RuntimeError("The immutable pretrained model revision was not resolved")\n    # Resolve a mutable branch exactly once, then pin every other asset to it.\n    options = {"revision": resolved, "trust_remote_code": False}\n    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n    if not getattr(tokenizer, "is_fast", False):\n        raise ValueError("Complete overflow windows require a fast tokenizer")\n    encoder = AutoModel.from_pretrained(model_id, config=config, **options)\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    if not getattr(encoder, "supports_gradient_checkpointing", False):\n        raise ValueError("The selected encoder must support gradient checkpointing")\n    encoder.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n    )\n    asset_dir = Path(asset_dir)\n    encoder_dir, tokenizer_dir = asset_dir / "encoder", asset_dir / "tokenizer"\n    encoder_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer_dir.mkdir(parents=True, exist_ok=True)\n    encoder.config.save_pretrained(encoder_dir)\n    tokenizer.save_pretrained(tokenizer_dir)\n    return encoder, tokenizer, resolved\n\n\ndef _binary_targets(targets, count):\n    values = torch.as_tensor(targets)\n    if values.shape != (count, NUM_LABELS):\n        raise ValueError(f"targets must have shape ({count}, {NUM_LABELS})")\n    if not bool(torch.isfinite(values).all()) or not bool(((values == 0) | (values == 1)).all()):\n        raise ValueError("targets must contain only finite binary values")\n    return values.to(dtype=torch.long)\n\n\ndef _turn_spans(text, separator):\n    """Locate nonempty raw turns in the unchanged separator-joined input."""\n    pieces = re.split(r"\\r\\n?|\\n", text)\n    delimiter = f" {separator} "\n    spans, cursor = [], 0\n    for piece in pieces:\n        if piece.strip():\n            spans.append((cursor, cursor + len(piece), len(spans)))\n        cursor += len(piece) + len(delimiter)\n    return delimiter.join(pieces), spans\n\n\nclass EncodeDataset(Dataset):\n    """Retain all fast-tokenizer overflow windows, including the final tokens."""\n\n    def __init__(self, texts, targets, tokenizer, max_length=512, overlap=64, *,\n                 utterance_evidence=False):\n        texts = list(texts)\n        if type(utterance_evidence) is not bool:\n            raise ValueError("utterance_evidence must be boolean")\n        if any(not isinstance(text, str) for text in texts):\n            raise ValueError("Every dialogue must be a text string")\n        for name, value in (("max_length", max_length), ("overlap", overlap)):\n            if isinstance(value, bool) or not isinstance(value, int):\n                raise ValueError(f"{name} must be an integer")\n        usable = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        if usable < 1 or not 0 <= overlap < usable:\n            raise ValueError("max_length must fit special tokens and 0 <= overlap < usable tokens")\n        if getattr(tokenizer, "pad_token_id", None) is None:\n            raise ValueError("The tokenizer must define a padding token")\n        separator = getattr(tokenizer, "sep_token", None)\n        if not isinstance(separator, str) or not separator:\n            raise ValueError("The tokenizer must define an utterance separator token")\n        self.tokenizer = tokenizer\n        self.utterance_evidence = utterance_evidence\n        self.targets = None if targets is None else _binary_targets(targets, len(texts)).clone()\n        self.documents = [[] for _ in texts]\n        # Unknown tokens still represent original content. Structural markers do\n        # not become local evidence, but remain in the unchanged global input.\n        structural_ids = set(getattr(tokenizer, "all_special_ids", []))\n        structural_ids.discard(getattr(tokenizer, "unk_token_id", None))\n        for start in range(0, len(texts), 128):\n            dialogues = [re.sub(r"\\r\\n?|\\n", f" {separator} ", text)\n                         for text in texts[start:start + 128]]\n            turn_spans = [_turn_spans(text, separator)[1] for text in texts[start:start + 128]] if utterance_evidence else None\n            options = {"return_offsets_mapping": True} if utterance_evidence else {}\n            encoded = tokenizer(\n                dialogues, truncation=True, max_length=max_length, stride=overlap,\n                return_overflowing_tokens=True, padding=False,\n                return_attention_mask=True,\n                **options,\n            )\n            if "overflow_to_sample_mapping" not in encoded:\n                raise ValueError("A fast tokenizer must return overflow_to_sample_mapping")\n            if "input_ids" not in encoded or "attention_mask" not in encoded:\n                raise ValueError("The tokenizer must return input_ids and attention_mask")\n            if utterance_evidence and "offset_mapping" not in encoded:\n                raise ValueError("Utterance evidence requires fast-tokenizer offset_mapping")\n            for index, source in enumerate(encoded["overflow_to_sample_mapping"]):\n                source = int(source)\n                if not 0 <= source < len(dialogues):\n                    raise ValueError("Tokenizer overflow mapping is out of range")\n                window = {key: list(encoded[key][index])\n                          for key in _MODEL_INPUTS if key in encoded}\n                length = len(window["input_ids"])\n                if not 0 < length <= max_length or any(len(value) != length for value in window.values()):\n                    raise ValueError("Tokenizer returned an invalid window length")\n                if not all(value in (0, 1) for value in window["attention_mask"]) or not any(window["attention_mask"]):\n                    raise ValueError("A real token window must have a binary, nonempty attention mask")\n                if utterance_evidence:\n                    offsets = encoded["offset_mapping"][index]\n                    if len(offsets) != length:\n                        raise ValueError("Tokenizer offsets must align with every token")\n                    spans = turn_spans[source]\n                    starts = [span[0] for span in spans]\n                    turn_ids, occurrence_keys = [], []\n                    for token, attended, offset in zip(window["input_ids"], window["attention_mask"], offsets):\n                        left, right = map(int, offset)\n                        if not 0 <= left <= right <= len(dialogues[source]):\n                            raise ValueError("Tokenizer offset is outside the prepared dialogue")\n                        position = bisect_right(starts, left) - 1\n                        valid = (bool(attended) and token not in structural_ids and right > left\n                                 and position >= 0 and right <= spans[position][1])\n                        turn_ids.append(spans[position][2] if valid else -1)\n                        occurrence_keys.append((left, right, int(token)) if valid else None)\n                    window["turn_ids"] = turn_ids\n                    window["_occurrence_keys"] = occurrence_keys\n                self.documents[start + source].append(window)\n        if any(not document for document in self.documents):\n            raise ValueError("Tokenizer returned a dialogue without any windows")\n        if utterance_evidence:\n            for document in self.documents:\n                counts = Counter(key for window in document for key in window["_occurrence_keys"] if key is not None)\n                for window in document:\n                    # Each original token has total mass one across overlapping\n                    # windows. Contextual occurrences are averaged, then tokens\n                    # within a turn are averaged; long/duplicated turns do not\n                    # gain extra attention entries. Adjacent turns are joined\n                    # only after this complete document-level reconstruction.\n                    window["token_weights"] = [1.0 / counts[key] if key is not None else 0.0\n                                               for key in window.pop("_occurrence_keys")]\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["targets"] = self.targets[index]\n        return item\n\n    def collate(self, examples):\n        """Pad tokens and windows; keep one target vector for each dialogue."""\n        if not examples or any(not item["windows"] for item in examples):\n            raise ValueError("A batch must contain nonempty dialogue windows")\n        keys = set(examples[0]["windows"][0])\n        if any(set(window) != keys for item in examples for window in item["windows"]):\n            raise ValueError("Token windows must use the same model input fields")\n        if any(("targets" in item) != ("targets" in examples[0]) for item in examples):\n            raise ValueError("A batch cannot mix labeled and unlabeled dialogues")\n        batch_size = len(examples)\n        window_count = max(len(item["windows"]) for item in examples)\n        token_count = max(len(window["input_ids"]) for item in examples for window in item["windows"])\n        batch = {key: torch.full(\n            (batch_size, window_count, token_count),\n            self.tokenizer.pad_token_id if key == "input_ids" else 0,\n            dtype=torch.long,\n        ) for key in _MODEL_INPUTS if key in keys}\n        if self.utterance_evidence:\n            batch["turn_ids"] = torch.full((batch_size, window_count, token_count), -1, dtype=torch.long)\n            batch["token_weights"] = torch.zeros((batch_size, window_count, token_count), dtype=torch.float32)\n        batch["window_mask"] = torch.zeros((batch_size, window_count), dtype=torch.bool)\n        for row, item in enumerate(examples):\n            for column, window in enumerate(item["windows"]):\n                length = len(window["input_ids"])\n                for key in keys:\n                    batch[key][row, column, :length] = torch.tensor(window[key], dtype=batch[key].dtype)\n                batch["window_mask"][row, column] = True\n        if "targets" in examples[0]:\n            batch["targets"] = torch.stack([item["targets"] for item in examples])\n        return batch\n\n\ndef validate_evidence_config(hidden=128, dropout=0.1):\n    if type(hidden) is not int or not 1 <= hidden <= 512:\n        raise ValueError("evidence_hidden must be an integer between 1 and 512")\n    if (isinstance(dropout, bool) or not isinstance(dropout, Real)\n            or not math.isfinite(dropout) or not 0 <= dropout < 1):\n        raise ValueError("evidence_dropout must be finite in [0, 1)")\n    return {"evidence_hidden": hidden, "evidence_dropout": float(dropout)}\n\n\nclass UtteranceEvidence(nn.Module):\n    """Learn an ordered neighboring-turn residual for the nausea head only."""\n\n    def __init__(self, encoder_hidden, global_hidden, hidden=128, dropout=0.1):\n        super().__init__()\n        validate_evidence_config(hidden, dropout)\n        with torch.random.fork_rng(devices=[]):\n            self.projection = nn.Sequential(nn.Linear(encoder_hidden * 3 + global_hidden, hidden),\n                                            nn.GELU(), nn.Dropout(dropout))\n            self.attention = nn.Linear(hidden, 1)\n            self.output = nn.Linear(hidden, 2)\n            nn.init.zeros_(self.output.weight)\n            nn.init.zeros_(self.output.bias)\n\n    def forward(self, turns, turn_mask, global_features):\n        if turns.ndim != 3 or turn_mask.shape != turns.shape[:2] or turn_mask.dtype != torch.bool:\n            raise ValueError("Turn representations require a matching boolean mask")\n        # Compact valid turns in their original chronological order. This also\n        # handles a rare turn composed entirely of structural special tokens.\n        compact = turns.new_zeros(turns.shape)\n        compact_mask = torch.zeros_like(turn_mask)\n        for row in range(len(turns)):\n            count = int(turn_mask[row].sum())\n            compact[row, :count] = turns[row, turn_mask[row]]\n            compact_mask[row, :count] = True\n        previous = torch.cat((torch.zeros_like(compact[:, :1]), compact[:, :-1]), dim=1)\n        following = torch.cat((compact[:, 1:], torch.zeros_like(compact[:, :1])), dim=1)\n        global_context = global_features.unsqueeze(1).expand(-1, compact.shape[1], -1)\n        evidence = self.projection(torch.cat((previous, compact, following, global_context), dim=-1))\n        scores = self.attention(evidence).squeeze(-1).float()\n        has_turn = compact_mask.any(1)\n        safe_mask = compact_mask.clone()\n        safe_mask[~has_turn, 0] = True\n        scores = scores.masked_fill(~safe_mask, -torch.inf)\n        attention = scores.softmax(1).to(evidence.dtype)\n        summary = (attention.unsqueeze(-1) * evidence).sum(1)\n        residual = self.output(summary)\n        return residual * has_turn.unsqueeze(-1).to(residual.dtype)\n\n\nclass DialogueClassifier(nn.Module):\n    """Train nine independent binary heads from all windows of each dialogue."""\n\n    def __init__(self, encoder, label_attention=False, contrastive=False, feature_dim=256, *,\n                 utterance_evidence=False, evidence_hidden=128, evidence_dropout=0.1):\n        super().__init__()\n        if isinstance(feature_dim, bool) or not isinstance(feature_dim, int) or feature_dim < 1:\n            raise ValueError("feature_dim must be a positive integer")\n        if not isinstance(label_attention, bool) or not isinstance(contrastive, bool):\n            raise ValueError("label_attention and contrastive must be boolean")\n        if type(utterance_evidence) is not bool:\n            raise ValueError("utterance_evidence must be boolean")\n        validate_evidence_config(evidence_hidden, evidence_dropout)\n        if utterance_evidence and (label_attention or contrastive):\n            raise ValueError("Utterance evidence requires the single global CE pathway")\n        self.encoder = encoder\n        self.label_attention = label_attention\n        self.contrastive = contrastive\n        self.encoder_window_batch = 8\n        hidden_size = int(encoder.config.hidden_size)\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, feature_dim), nn.GELU())\n        self.dropout = nn.Dropout(0.1)\n        if label_attention:\n            self.token_queries = nn.Parameter(torch.empty(NUM_LABELS, hidden_size))\n            self.window_queries = nn.Parameter(torch.empty(NUM_LABELS, feature_dim))\n            nn.init.normal_(self.token_queries, std=0.02)\n            nn.init.normal_(self.window_queries, std=0.02)\n        else:\n            self.register_parameter("token_queries", None)\n            self.register_parameter("window_queries", None)\n        self.class_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, 2))\n        self.class_bias = nn.Parameter(torch.zeros(NUM_LABELS, 2))\n        self.feature_weights = nn.Parameter(torch.empty(NUM_LABELS, feature_dim, CONTRASTIVE_DIM))\n        self.feature_bias = nn.Parameter(torch.zeros(NUM_LABELS, CONTRASTIVE_DIM))\n        nn.init.xavier_uniform_(self.class_weights)\n        nn.init.xavier_uniform_(self.feature_weights)\n        self.evidence = (UtteranceEvidence(hidden_size, feature_dim, evidence_hidden, evidence_dropout)\n                         if utterance_evidence else None)\n\n    def forward(self, batch):\n        window_mask = batch.get("window_mask")\n        ids, token_mask = batch.get("input_ids"), batch.get("attention_mask")\n        if not isinstance(ids, torch.Tensor) or ids.ndim != 3:\n            raise ValueError("input_ids must have shape (batch, windows, tokens)")\n        if not isinstance(token_mask, torch.Tensor) or token_mask.shape != ids.shape:\n            raise ValueError("attention_mask must match input_ids")\n        if not isinstance(window_mask, torch.Tensor) or window_mask.shape != ids.shape[:2] or window_mask.dtype != torch.bool:\n            raise ValueError("window_mask must be a boolean (batch, windows) tensor")\n        if ids.shape[0] < 1 or not bool(window_mask.any(dim=1).all()):\n            raise ValueError("Every dialogue must contain at least one real window")\n        if not bool(((token_mask == 0) | (token_mask == 1)).all()):\n            raise ValueError("attention_mask must be binary")\n        if not bool((token_mask.sum(-1)[window_mask] > 0).all()):\n            raise ValueError("Real windows must contain attended tokens")\n        chunk_size = self.encoder_window_batch\n        if isinstance(chunk_size, bool) or not isinstance(chunk_size, int) or chunk_size < 1:\n            raise ValueError("encoder_window_batch must be a positive integer")\n        flat_mask = window_mask.flatten()\n        turn_sum = turn_mass = None\n        if self.evidence is not None:\n            turn_ids, token_weights = batch.get("turn_ids"), batch.get("token_weights")\n            if (not isinstance(turn_ids, torch.Tensor) or turn_ids.shape != ids.shape or turn_ids.dtype != torch.long\n                    or not isinstance(token_weights, torch.Tensor) or token_weights.shape != ids.shape\n                    or not token_weights.is_floating_point()):\n                raise ValueError("Utterance evidence requires aligned turn_ids and token_weights")\n            valid_turn_tokens = turn_ids >= 0\n            if (bool((turn_ids < -1).any()) or not bool(torch.isfinite(token_weights).all())\n                    or bool((token_weights < 0).any()) or bool((token_weights > 1).any())\n                    or bool((token_weights[valid_turn_tokens] <= 0).any())\n                    or bool((token_weights[~valid_turn_tokens] != 0).any())\n                    or bool((valid_turn_tokens & ~token_mask.bool()).any())\n                    or bool((valid_turn_tokens & ~window_mask.unsqueeze(-1)).any())):\n                raise ValueError("Invalid turn IDs, padding masks, or overlap occurrence weights")\n            turn_count = max(1, int(turn_ids.max()) + 1)\n            if turn_count > int(token_mask.sum()):\n                raise ValueError("Turn IDs exceed the available token positions")\n            selected_turn_ids = turn_ids.flatten(0, 1)[flat_mask]\n            selected_weights = token_weights.flatten(0, 1)[flat_mask]\n            document_ids = torch.arange(ids.shape[0], device=ids.device).unsqueeze(1).expand_as(window_mask).flatten()[flat_mask]\n            turn_sum = torch.zeros((ids.shape[0] * turn_count, self.encoder.config.hidden_size),\n                                   dtype=torch.float32, device=ids.device)\n            turn_mass = torch.zeros(ids.shape[0] * turn_count, dtype=torch.float32, device=ids.device)\n        inputs = {}\n        for key in _MODEL_INPUTS:\n            if key in batch:\n                if batch[key].shape != ids.shape:\n                    raise ValueError(f"{key} must match input_ids")\n                inputs[key] = batch[key].flatten(0, 1)[flat_mask]\n        representations = []\n        for start in range(0, len(inputs["input_ids"]), chunk_size):\n            chunk = {key: value[start:start + chunk_size] for key, value in inputs.items()}\n            hidden = self.encoder(**chunk).last_hidden_state\n            if not bool(torch.isfinite(hidden).all()):\n                raise FloatingPointError("Encoder produced nonfinite hidden states")\n            if self.evidence is not None:\n                local_ids = selected_turn_ids[start:start + chunk_size]\n                local_weights = selected_weights[start:start + chunk_size]\n                present = local_ids >= 0\n                rows = document_ids[start:start + chunk_size].unsqueeze(1) * turn_count + local_ids.clamp_min(0)\n                turn_sum = turn_sum.index_add(0, rows[present], hidden.float()[present] * local_weights[present, None])\n                turn_mass = turn_mass.index_add(0, rows[present], local_weights[present])\n            if self.label_attention:\n                token_scores = torch.einsum("nth,lh->ntl", hidden.float(), self.token_queries.float())\n                token_scores = token_scores / math.sqrt(hidden.shape[-1])\n                token_scores = token_scores.masked_fill(~chunk["attention_mask"].bool().unsqueeze(-1), -torch.inf)\n                token_attention = torch.softmax(token_scores, dim=1).to(hidden.dtype)\n                context = torch.einsum("ntl,nth->nlh", token_attention, hidden)\n                cls = hidden[:, 0].unsqueeze(1).expand(-1, NUM_LABELS, -1)\n                representations.append(self.projection(torch.cat((cls, context), dim=-1)))\n            else:\n                mask = chunk["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n                mean = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1)\n                representations.append(self.projection(torch.cat((hidden[:, 0], mean), dim=-1)))\n        valid = self.dropout(torch.cat(representations, dim=0))\n        if not bool(torch.isfinite(valid).all()):\n            raise FloatingPointError("Window projection produced nonfinite features")\n        padded_shape = (*window_mask.shape, *valid.shape[1:])\n        windows = valid.new_zeros(padded_shape)\n        windows = windows.flatten(0, 1).index_copy(\n            0, flat_mask.nonzero(as_tuple=False).flatten(), valid,\n        ).reshape(padded_shape)\n        if self.label_attention:\n            scores = (windows.float() * self.window_queries.float()[None, None]).sum(-1)\n            scores = scores / math.sqrt(windows.shape[-1])\n            scores = scores.masked_fill(~window_mask.unsqueeze(-1), -torch.inf)\n            attention = torch.softmax(scores, dim=1).to(windows.dtype)\n            pooled = (attention.unsqueeze(-1) * windows).sum(dim=1)\n        else:\n            pooled = windows.sum(dim=1) / window_mask.sum(dim=1, keepdim=True).to(windows.dtype)\n            global_features = pooled\n            pooled = pooled.unsqueeze(1).expand(-1, NUM_LABELS, -1)\n        pooled = self.dropout(pooled)\n        logits = torch.einsum("bld,ldc->blc", pooled, self.class_weights) + self.class_bias\n        features = torch.einsum("bld,ldf->blf", pooled, self.feature_weights) + self.feature_bias\n        features = F.normalize(features.float(), dim=-1)\n        if self.evidence is not None:\n            turns = (turn_sum / turn_mass.clamp_min(1).unsqueeze(-1)).reshape(ids.shape[0], turn_count, -1)\n            turn_mask = turn_mass.reshape(ids.shape[0], turn_count) > 0\n            # No new stochastic operation runs until the original global logits\n            # and features are complete. All other symptom logits are untouched.\n            residual = self.evidence(turns.to(global_features.dtype), turn_mask, global_features)\n            nausea_mask = logits.new_zeros((1, NUM_LABELS, 1))\n            nausea_mask[:, 6] = 1\n            logits = logits + nausea_mask * residual.unsqueeze(1)\n        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(features).all()):\n            raise FloatingPointError("Classifier produced nonfinite logits or features")\n        return logits, features\n\n\ndef predictions(logits):\n    """The learned head\'s winning class; no fitted post-training cutoff."""\n    if logits.ndim != 3 or logits.shape[1:] != (NUM_LABELS, 2):\n        raise ValueError("logits must have shape (batch, 9, 2)")\n    if not bool(torch.isfinite(logits).all()):\n        raise FloatingPointError("Cannot predict from nonfinite logits")\n    return logits.argmax(dim=-1)\n\n\ndef classification_loss(logits, targets, nausea_positive_weight=1.0, *,\n                        positive_class_weights=None):\n    """Weight positive CE items by label, retaining the batch-by-label mean.\n\n    The scalar nausea weight remains supported for existing callers. Weights\n    affect supervised training only; prediction still uses the raw argmax.\n    """\n    if isinstance(nausea_positive_weight, bool) or not isinstance(nausea_positive_weight, Real):\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    try:\n        positive_weight = float(nausea_positive_weight)\n    except (OverflowError, ValueError) as error:\n        raise ValueError("nausea_positive_weight must be a finite positive real number") from error\n    if not math.isfinite(positive_weight) or positive_weight <= 0:\n        raise ValueError("nausea_positive_weight must be a finite positive real number")\n    if positive_class_weights is None:\n        class_weights = [1.0] * NUM_LABELS\n        class_weights[6] = positive_weight\n    else:\n        if positive_weight != 1.0:\n            raise ValueError("Use positive_class_weights or a nondefault nausea_positive_weight, not both")\n        if not isinstance(positive_class_weights, (list, tuple)) or len(positive_class_weights) != NUM_LABELS:\n            raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n        class_weights = []\n        for value in positive_class_weights:\n            if isinstance(value, bool) or not isinstance(value, Real):\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            try:\n                weight = float(value)\n            except (OverflowError, ValueError) as error:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers") from error\n            if not math.isfinite(weight) or weight <= 0:\n                raise ValueError("positive_class_weights must contain exactly nine finite positive real numbers")\n            class_weights.append(weight)\n    predictions(logits)\n    targets = _binary_targets(targets, len(logits)).to(logits.device)\n    if len(logits) < 1:\n        raise ValueError("classification_loss requires a nonempty batch")\n    if all(weight == 1.0 for weight in class_weights):\n        loss = F.cross_entropy(logits.float().reshape(-1, 2), targets.reshape(-1))\n    else:\n        per_item = F.cross_entropy(\n            logits.float().reshape(-1, 2), targets.reshape(-1), reduction="none",\n        ).reshape(-1, NUM_LABELS)\n        positive_weights = per_item.new_tensor(class_weights).unsqueeze(0)\n        weights = torch.where(targets == 1, positive_weights, 1.0)\n        loss = (per_item * weights).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Classification loss is nonfinite")\n    return loss\n\n\ndef binary_supervised_contrastive_loss(features1, features2, targets, temperature=0.1):\n    """Binary supervised contrastive loss per symptom, with balanced anchors.\n\n    The two views come from independent dropout forwards of the same dialogues.\n    Same-class examples are positives, self matches are excluded, and each\n    present binary class receives equal anchor weight within each symptom.\n    Thus the many symptom-negative examples cannot dominate the anchor average.\n    """\n    if features1.ndim != 3 or features1.shape[1] != NUM_LABELS or features1.shape != features2.shape:\n        raise ValueError("Both feature views must have matching (batch, 9, feature_dim) shapes")\n    if features1.shape[0] < 1 or features1.shape[2] < 1:\n        raise ValueError("Contrastive loss requires nonempty feature views")\n    if not bool(torch.isfinite(features1).all()) or not bool(torch.isfinite(features2).all()):\n        raise FloatingPointError("Contrastive features must be finite")\n    if isinstance(temperature, bool) or not math.isfinite(float(temperature)) or temperature <= 0:\n        raise ValueError("temperature must be positive and finite")\n    labels = _binary_targets(targets, len(features1)).to(features1.device)\n    features = F.normalize(torch.cat((features1.float(), features2.float()), dim=0), dim=-1)\n    labels = torch.cat((labels, labels), dim=0)\n    self_mask = torch.eye(len(features), dtype=torch.bool, device=features.device)\n    losses = []\n    for label in range(NUM_LABELS):\n        scores = features[:, label] @ features[:, label].T / float(temperature)\n        denominator = torch.logsumexp(scores.masked_fill(self_mask, -torch.inf), dim=1, keepdim=True)\n        log_probability = scores - denominator\n        positive = (labels[:, label, None] == labels[None, :, label]) & ~self_mask\n        anchor_loss = -(log_probability * positive).sum(1) / positive.sum(1)\n        class_losses = [anchor_loss[labels[:, label] == value].mean()\n                        for value in (0, 1) if bool((labels[:, label] == value).any())]\n        losses.append(torch.stack(class_losses).mean())\n    loss = torch.stack(losses).mean()\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("Supervised contrastive loss is nonfinite")\n    return loss\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision always enforces the actual invocation deadline.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining > 0 and estimate * self.safety_factor <= remaining\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and a bounded wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        returncode = process.wait(timeout=max(0.0, deadline - margin - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline)\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise\n    if returncode:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'runtime.py': '"""Training-only, single-view symptom experiments within one shared time budget."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import data\nfrom .bootstrap import prepare_data\nfrom .budget import Budget, BudgetExceeded\n\n\nPOSITIVE_CLASS_WEIGHTS = [1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]\nCANDIDATES = (\n    {"candidate_id": "baseline_ce", "utterance_evidence": False},\n    {"candidate_id": "utterance_evidence", "utterance_evidence": True},\n)\n\n\ndef _stamp():\n    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n\n\ndef _read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef _digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     allow_nan=False).encode("utf-8")).hexdigest()\n\n\ndef dataset_digest(texts, targets):\n    digest = hashlib.sha256()\n    for text, target in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(target, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _safe(root, relative):\n    candidate = root / relative\n    path = candidate.resolve()\n    if not path.is_relative_to(root.resolve()) or path == root.resolve():\n        raise ValueError("Artifact path must stay inside its run directory")\n    for part in (candidate, *candidate.parents):\n        if part == root.parent:\n            break\n        if part.is_symlink():\n            raise ValueError("Symlink artifacts are not supported")\n    return path\n\n\ndef _verify_hashes(root, hashes, budget=None):\n    if not isinstance(hashes, dict) or not hashes:\n        raise ValueError("Completed artifacts require nonempty hashes")\n    for relative, expected in hashes.items():\n        if budget:\n            budget.check("verify saved artifacts")\n        path = _safe(root, relative)\n        if not path.is_file() or _hash(path) != expected:\n            raise ValueError("Completed artifact is missing or changed: " + relative)\n\n\ndef _model_hashes(root, manifest, extras=()):\n    assets = _safe(root, manifest["assets"])\n    weights = _safe(root, manifest["weights"])\n    if not weights.is_file() or not assets.is_dir():\n        raise FileNotFoundError("Model weights or assets are missing")\n    files = [weights, *sorted(p for p in assets.rglob("*") if p.is_file()),\n             *(_safe(root, name) for name in extras)]\n    if not any(p.parent.name == "tokenizer" for p in files):\n        raise FileNotFoundError("Tokenizer assets are missing")\n    return {p.relative_to(root.resolve()).as_posix(): _hash(p) for p in files}\n\n\ndef _publish_tree(source, destination, budget):\n    """Copy immutable local artifacts; caller writes completion marker last."""\n    destination.mkdir(parents=True, exist_ok=False)\n    for path in sorted(source.rglob("*")):\n        if not path.is_file():\n            continue\n        budget.check("publish completed model")\n        target = destination / path.relative_to(source)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        temporary = target.with_name(target.name + ".copying")\n        with path.open("rb") as reader, temporary.open("wb") as writer:\n            while block := reader.read(4 * 1024 * 1024):\n                budget.check("copy model to output storage")\n                writer.write(block)\n        if _hash(path) != _hash(temporary):\n            raise OSError("Published model copy failed its content hash check")\n        temporary.replace(target)\n\n\ndef validate_settings(settings):\n    result = dict(settings)\n    if result.get("version") != "ver18" or result.get("backend") != "neural":\n        raise ValueError("This runner supports ver18 neural experiments only")\n    if result.get("label_attention", False) or result.get("contrastive_weight", 0) != 0:\n        raise ValueError("ver18 uses a single CE view without attention or contrastive loss")\n    if result.get("refit_full_training") is not True:\n        raise ValueError("Full refit is requested and skipped only when the time budget cannot fit it")\n    for key in ("batch_size", "encoder_window_batch", "gradient_accumulation", "max_epochs", "patience", "max_length", "log_every"):\n        if type(result.get(key)) is not int or result[key] < 1:\n            raise ValueError(key + " must be a positive integer")\n    if not 1 <= result["max_epochs"] <= 3:\n        raise ValueError("ver18 supports at most three selection epochs")\n    horizon = result.get("scheduler_horizon_epochs", 4)\n    if type(horizon) is not int or horizon != 4:\n        raise ValueError("ver18 keeps its matched four-epoch scheduler horizon")\n    result["scheduler_horizon_epochs"] = horizon\n    result.update(_architecture(result))\n    if type(result.get("seed")) is not int or result["seed"] < 0:\n        raise ValueError("seed must be a nonnegative integer")\n    if type(result.get("overlap")) is not int or not 0 <= result["overlap"] < result["max_length"] - 2:\n        raise ValueError("overlap must fit within a token window")\n    if not 0 < result.get("holdout_size", 0) < .5:\n        raise ValueError("holdout_size must lie in (0, .5)")\n    for key in ("learning_rate", "head_learning_rate", "time_budget_minutes", "budget_safety_factor", "finalization_reserve_seconds"):\n        value = result.get(key)\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:\n            raise ValueError(key + " must be positive and finite")\n    if result["time_budget_minutes"] > 180:\n        raise ValueError("The execution budget cannot exceed 180 minutes")\n    if result["finalization_reserve_seconds"] >= result["time_budget_minutes"] * 60:\n        raise ValueError("Finalization reserve must be smaller than the total budget")\n    if result["budget_safety_factor"] < 1:\n        raise ValueError("budget_safety_factor must be at least one")\n    if not isinstance(result.get("resume_run", ""), str):\n        raise ValueError("resume_run must be a run-directory path or empty string")\n    result.update(label_attention=False, contrastive_weight=0.0, ce_views=1)\n    return result\n\n\ndef _stable_settings(settings):\n    excluded = {"deadline_monotonic", "input_zip", "train_dir", "validation_dir", "work_dir", "output_dir",\n                "resume_run", "log_every", "time_budget_minutes", "finalization_reserve_seconds", "budget_safety_factor"}\n    return {key: value for key, value in settings.items() if key not in excluded}\n\n\ndef _loader(texts, targets, tokenizer, settings, shuffle):\n    from torch.utils.data import DataLoader\n    from .neural import EncodeDataset\n    dataset = EncodeDataset(texts, targets, tokenizer, settings["max_length"], settings["overlap"],\n                            utterance_evidence=settings.get("utterance_evidence", False))\n    return DataLoader(dataset, batch_size=settings["batch_size"], shuffle=shuffle,\n                      num_workers=0, collate_fn=dataset.collate)\n\n\ndef _save_weights(model, path):\n    import torch\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, temporary)\n    temporary.replace(path)\n\n\ndef _load_weights(model, path):\n    import torch\n    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)\n\n\ndef _predict(model, loader, device, budget=None, training=False):\n    import torch\n    from .neural import predictions\n    result = []\n    model.eval()\n    with torch.inference_mode():\n        for batch in loader:\n            if budget:\n                budget.check("evaluate model", training=training)\n            logits, _ = model({key: value.to(device) for key, value in batch.items()})\n            result.append(predictions(logits).cpu().numpy())\n    if not result:\n        raise ValueError("Cannot predict an empty dataset")\n    return np.concatenate(result).astype(np.int8)\n\n\ndef _optimizer(model, settings, batches, device):\n    import torch\n    encoder = list(model.encoder.parameters())\n    identifiers = {id(p) for p in encoder}\n    heads = [p for p in model.parameters() if id(p) not in identifiers]\n    optimizer = torch.optim.AdamW([\n        {"params": encoder, "lr": settings["learning_rate"]},\n        {"params": heads, "lr": settings["head_learning_rate"]}], weight_decay=.01)\n    updates = math.ceil(batches / settings["gradient_accumulation"]) * settings.get("scheduler_horizon_epochs", 4)\n    warmup = max(1, int(updates * .1))\n    def schedule(step):\n        return (step + 1) / warmup if step < warmup else max(0., (updates - step) / max(1, updates - warmup))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n    cuda = device.type == "cuda"\n    dtype = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else torch.float16\n    scaler = torch.amp.GradScaler("cuda", enabled=cuda and dtype == torch.float16)\n    return optimizer, scheduler, scaler, dtype\n\n\ndef _train_epoch(model, loader, settings, device, state, budget, epoch):\n    import torch\n    from .neural import classification_loss\n    optimizer, scheduler, scaler, dtype = state\n    accumulation = settings["gradient_accumulation"]\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    loss_total, counts = 0., np.zeros(9, dtype=np.int64)\n    started = time.monotonic()\n    for step, batch in enumerate(loader):\n        budget.check("train microbatch", training=True)\n        counts += batch["targets"].sum(0).numpy()\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda"):\n            logits, _ = model(batch)\n            loss = classification_loss(logits, batch["targets"], positive_class_weights=settings["positive_class_weights"])\n        if not bool(torch.isfinite(loss)):\n            raise FloatingPointError("Nonfinite training loss")\n        group_start = step // accumulation * accumulation\n        group_size = min(accumulation, len(loader) - group_start)\n        scaler.scale(loss / group_size).backward()\n        loss_total += float(loss.detach())\n        if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            optimizer.zero_grad(set_to_none=True)\n        if (step + 1) % settings["log_every"] == 0:\n            print(f"{settings[\'candidate_id\']} epoch {epoch}: {step + 1}/{len(loader)} batches; "\n                  f"{budget.remaining() / 60:.1f} minutes left", flush=True)\n    return {"epoch": epoch, "loss": loss_total / len(loader), "classification_loss": loss_total / len(loader),\n            "ce_views": 1, "scheduler_horizon_epochs": settings.get("scheduler_horizon_epochs", 4),\n            "positive_target_counts": counts.tolist(), "training_seconds": time.monotonic() - started}\n\n\ndef _fit_selection(model, fit_loader, hold_loader, truth, indices, settings, device, local, budget, full_ratio):\n    state = _optimizer(model, settings, len(fit_loader), device)\n    best, stale, history, stop = None, 0, [], "max_epochs"\n    for epoch in range(1, settings["max_epochs"] + 1):\n        if history:\n            next_cost = max(row["epoch_seconds"] for row in history)\n            refit_cost = max(row["training_seconds"] for row in history) * full_ratio * epoch + 120\n            if not budget.can_start(next_cost + refit_cost, training=True):\n                # Prior runs improved chiefly at epoch three. Prefer reaching that\n                # point on the fit partition when a fresh full refit cannot fit.\n                minimum_useful_epochs = min(3, settings["max_epochs"])\n                if epoch > minimum_useful_epochs or not budget.can_start(next_cost, training=True):\n                    stop = "budget_reserving_refit"\n                    break\n        try:\n            started = time.monotonic()\n            row = _train_epoch(model, fit_loader, settings, device, state, budget, epoch)\n            evaluate_started = time.monotonic()\n            predicted = _predict(model, hold_loader, device, budget, training=True)\n            row["evaluation_seconds"] = time.monotonic() - evaluate_started\n            metrics = data.score_predictions(truth, predicted)\n            row["holdout_macro_f1"], row["holdout_metrics"] = metrics["macro_f1"], metrics\n            save_started = time.monotonic()\n            if best is None or metrics["macro_f1"] > best["metrics"]["macro_f1"]:\n                budget.check("save selected epoch")\n                _save_weights(model, local / "selection_weights.pt")\n                np.savez_compressed(local / "holdout_predictions.npz", indices=indices, targets=truth, predictions=predicted)\n                best, stale = {"selected_epochs": epoch, "metrics": metrics}, 0\n            else:\n                stale += 1\n            row["save_seconds"] = time.monotonic() - save_started\n            row["epoch_seconds"] = time.monotonic() - started\n            history.append(row)\n            data.write_json(local / "selection_history.json", history)\n            print(f"{settings[\'candidate_id\']} Training epoch {epoch}: Macro F1={metrics[\'macro_f1\']:.6f}; "\n                  f"{row[\'epoch_seconds\'] / 60:.1f} minutes", flush=True)\n            if stale >= settings["patience"]:\n                stop = "early_stopping"\n                break\n        except BudgetExceeded:\n            stop = "budget_during_epoch"\n            break\n    if best is None:\n        raise BudgetExceeded("No complete Training epoch and holdout evaluation fit the budget")\n    return {**best, "history": history, "stop_reason": stop}\n\n\ndef _manifest(settings, revision, epochs, samples, scope):\n    return {"format": "mission3_categorical_v1", "kind": "neural", "version": "ver18",\n            "labels": list(data.TARGET_SYMPTOMS), "decision_rule": "categorical predict/argmax",\n            "model_id": settings["model_id"], "resolved_revision": revision,\n            "selected_epochs": epochs, "fitted_samples": samples, "training_scope": scope,\n            "positive_class_weights": settings["positive_class_weights"], "label_attention": False, "contrastive": False,\n            **_architecture(settings), "scheduler_horizon_epochs": settings.get("scheduler_horizon_epochs", 4),\n            "max_length": settings["max_length"], "overlap": settings["overlap"],\n            "batch_size": settings["batch_size"], "encoder_window_batch": settings["encoder_window_batch"],\n            "assets": "assets", "weights": "selection_weights.pt" if scope == "fit_only" else "final_weights.pt"}\n\n\ndef _architecture(settings):\n    from .neural import validate_evidence_config\n    enabled = settings.get("utterance_evidence", False)\n    if type(enabled) is not bool:\n        raise ValueError("utterance_evidence must be boolean")\n    return {"utterance_evidence": enabled,\n            **validate_evidence_config(settings.get("evidence_hidden", 128),\n                                       settings.get("evidence_dropout", .1))}\n\n\ndef _device(settings):\n    import torch\n    if not torch.cuda.is_available() and not settings.get("allow_cpu", False):\n        raise RuntimeError("Select a Colab GPU runtime before starting ver18")\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef train_candidate(texts, targets, fit, holdout, settings, local, budget):\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything\n    device = _device(settings)\n    seed_everything(settings["seed"])\n    budget.check("prepare candidate encoder", training=True)\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], local / "assets")\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False, **_architecture(settings))\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    try:\n        budget.check("tokenize candidate", training=True)\n        fit_loader = _loader([texts[i] for i in fit], targets[fit], tokenizer, settings, True)\n        hold_loader = _loader([texts[i] for i in holdout], targets[holdout], tokenizer, settings, False)\n        result = _fit_selection(model, fit_loader, hold_loader, targets[holdout], holdout, settings, device,\n                                local, budget, len(targets) / len(fit))\n        manifest = _manifest(settings, revision, result["selected_epochs"], len(fit), "fit_only")\n        data.write_json(local / "candidate_model.ckpt", manifest)\n        data.write_json(local / "training_metrics.json", result["metrics"])\n        return manifest, result\n    finally:\n        del model, encoder\n        _release()\n\n\ndef refit_selected(texts, targets, settings, epochs, local, budget):\n    """Only a fully completed fresh refit is eligible for publication."""\n    from .neural import DialogueClassifier, prepare_encoder, seed_everything\n    if type(epochs) is not int or not 1 <= epochs <= settings["max_epochs"] <= 3:\n        raise ValueError("Refit epochs must match a completed selection epoch between one and three")\n    device = _device(settings)\n    seed_everything(settings["seed"])\n    budget.check("prepare fresh refit", training=True)\n    encoder, tokenizer, revision = prepare_encoder(settings["model_id"], settings["revision"], local / "assets")\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False, **_architecture(settings))\n    model.encoder_window_batch = settings["encoder_window_batch"]\n    model.to(device)\n    try:\n        loader = _loader(texts, targets, tokenizer, settings, True)\n        state = _optimizer(model, settings, len(loader), device)\n        history = []\n        for epoch in range(1, epochs + 1):\n            row = _train_epoch(model, loader, settings, device, state, budget, epoch)\n            history.append(row)\n            data.write_json(local / "final_history.json", history)\n            print(f"Fresh all-Training refit epoch {epoch}/{epochs}: loss={row[\'loss\']:.5f}", flush=True)\n        budget.check("save completed full refit")\n        _save_weights(model, local / "final_weights.pt")\n        probe = _predict(model, _loader(texts[:12], None, tokenizer, settings, False), device, budget)\n        manifest = _manifest(settings, revision, epochs, len(targets), "all_training")\n        data.write_json(local / "final_model.ckpt", manifest)\n        return manifest, probe\n    finally:\n        del model, encoder\n        _release()\n\n\ndef predict_checkpoint(checkpoint, texts, budget=None):\n    import torch\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\n    from .neural import DialogueClassifier\n    checkpoint = Path(checkpoint)\n    manifest, root = _read(checkpoint), checkpoint.parent\n    if manifest.get("format") != "mission3_categorical_v1" or tuple(manifest["labels"]) != data.TARGET_SYMPTOMS:\n        raise ValueError("Unsupported checkpoint or symptom ordering")\n    if manifest["kind"] != "neural" or manifest["label_attention"] or manifest["contrastive"] or manifest.get("version") != "ver18":\n        raise ValueError("ver18 inference requires its single-view classifier")\n    if not {"utterance_evidence", "evidence_hidden", "evidence_dropout"} <= manifest.keys():\n        raise ValueError("ver18 checkpoint must declare its utterance evidence architecture")\n    architecture = _architecture(manifest)\n    if budget:\n        budget.check("load final checkpoint")\n    assets = _safe(root, manifest["assets"])\n    config = AutoConfig.from_pretrained(assets / "encoder", local_files_only=True, trust_remote_code=False)\n    encoder = AutoModel.from_config(config, trust_remote_code=False)\n    tokenizer = AutoTokenizer.from_pretrained(assets / "tokenizer", local_files_only=True, trust_remote_code=False)\n    model = DialogueClassifier(encoder, label_attention=False, contrastive=False, **architecture)\n    model.encoder_window_batch = manifest["encoder_window_batch"]\n    _load_weights(model, _safe(root, manifest["weights"]))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device)\n    try:\n        return _predict(model, _loader(texts, None, tokenizer, manifest, False), device, budget)\n    finally:\n        del model, encoder\n        _release()\n\n\ndef inference(checkpoint, label_dir, output, budget=None):\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    predicted = predict_checkpoint(checkpoint, texts, budget)\n    if budget:\n        budget.check("save final predictions")\n    data.write_predictions(paths, predicted, output)\n    return paths, predicted\n\n\ndef _candidate_record(settings, texts, targets, fit, holdout, root, budget):\n    name = settings["candidate_id"]\n    directory = root / "experiments" / name\n    completed = directory / "completed.json"\n    stable = _stable_settings(settings)\n    if completed.is_file():\n        record = _read(completed)\n        if record["settings"] != stable or record["settings_digest"] != _digest(stable) or record["candidate_id"] != name:\n            raise ValueError("Saved candidate configuration changed")\n        if Path(record["directory"]).parts[:2] != ("experiments", name):\n            raise ValueError("Saved candidate directory changed")\n        _verify_hashes(root, record["artifact_hashes"], budget)\n        original = _read(_safe(root, record["directory"] + "/candidate_model.ckpt"))\n        expected = {**original, "weights": record["directory"] + "/" + original["weights"],\n                    "assets": record["directory"] + "/" + original["assets"]}\n        if expected != record["manifest"]:\n            raise ValueError("Saved candidate manifest changed")\n        print("Reusing completed Training candidate:", name, flush=True)\n    else:\n        started, attempt = time.monotonic(), _stamp()\n        local = Path(settings["work_dir"]) / "training" / root.name / name / attempt\n        local.mkdir(parents=True, exist_ok=False)\n        data.write_json(local / "settings.json", stable)\n        manifest, result = train_candidate(texts, targets, fit, holdout, settings, local, budget)\n        destination = directory / attempt\n        _publish_tree(local, destination, budget)\n        relative = destination.relative_to(root).as_posix()\n        root_manifest = {**manifest, "weights": relative + "/" + manifest["weights"],\n                         "assets": relative + "/" + manifest["assets"]}\n        record = {"candidate_id": name, "directory": relative, "settings": stable,\n                  "settings_digest": _digest(stable), "manifest": root_manifest,\n                  "metrics": result["metrics"], "history": result["history"],\n                  "stop_reason": result["stop_reason"], "seconds": time.monotonic() - started}\n        record["artifact_hashes"] = _model_hashes(root, root_manifest, (\n            relative + "/holdout_predictions.npz", relative + "/selection_history.json",\n            relative + "/candidate_model.ckpt", relative + "/training_metrics.json", relative + "/settings.json"))\n        budget.check("commit completed candidate")\n        data.write_json(completed, record)\n    with np.load(_safe(root, record["directory"] + "/holdout_predictions.npz"), allow_pickle=False) as saved:\n        if not np.array_equal(saved["indices"], holdout) or not np.array_equal(saved["targets"], targets[holdout]):\n            raise ValueError("Saved candidate Training targets or split changed")\n        if data.score_predictions(targets[holdout], saved["predictions"]) != record["metrics"]:\n            raise ValueError("Saved candidate scores disagree with discrete predictions")\n    if record["history"] != _read(_safe(root, record["directory"] + "/selection_history.json")):\n        raise ValueError("Saved timing or epoch history changed")\n    return record\n\n\ndef _refit_estimate(record, full_ratio):\n    return (max(row["training_seconds"] for row in record["history"]) * full_ratio\n            * record["manifest"]["selected_epochs"] + 120)\n\n\ndef _write_table(root, records):\n    columns = ["candidate", "selected_epochs", "macro_f1", "nausea_f1", "nausea_precision", "nausea_recall",\n               "seconds", "stop_reason", "positive_class_weights", "utterance_evidence"]\n    temporary = root / "candidate_summary.csv.tmp"\n    with temporary.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns)\n        writer.writeheader()\n        for record in records:\n            nausea = record["metrics"]["per_symptom"][data.TARGET_SYMPTOMS[6]]\n            writer.writerow({"candidate": record["candidate_id"], "selected_epochs": record["manifest"]["selected_epochs"],\n                             "macro_f1": record["metrics"]["macro_f1"], "nausea_f1": nausea["f1"],\n                             "nausea_precision": nausea["precision"], "nausea_recall": nausea["recall"],\n                             "seconds": record["seconds"], "stop_reason": record["stop_reason"],\n                             "positive_class_weights": json.dumps(record["settings"]["positive_class_weights"]),\n                             "utterance_evidence": record["manifest"]["utterance_evidence"]})\n    temporary.replace(root / "candidate_summary.csv")\n\n\ndef _write_comparison(root, records, selected, budget):\n    """Aggregate paired Training predictions without transcript or row exports."""\n    budget.check("compare completed Training candidates")\n    by_name = {record["candidate_id"]: record for record in records}\n    report = {\n        "scope": "Training internal holdout only; not official Validation or full-Training fit metrics",\n        "selection_rule": "Highest Training holdout Macro F1; exact ties keep the first declared candidate",\n        "selected_candidate": selected["candidate_id"],\n        "completed_candidates": list(by_name),\n        "comparison_available": all(name in by_name for name in ("baseline_ce", "utterance_evidence")),\n    }\n    if report["comparison_available"]:\n        arrays = []\n        for name in ("baseline_ce", "utterance_evidence"):\n            path = _safe(root, by_name[name]["directory"] + "/holdout_predictions.npz")\n            with np.load(path, allow_pickle=False) as saved:\n                arrays.append({key: saved[key].copy() for key in ("indices", "targets", "predictions")})\n        baseline, challenger = arrays\n        if not np.array_equal(baseline["indices"], challenger["indices"]) or not np.array_equal(baseline["targets"], challenger["targets"]):\n            raise ValueError("Candidate comparison requires identical Training holdout rows and targets")\n        truth = baseline["targets"]\n        before, after = baseline["predictions"], challenger["predictions"]\n        scores = [data.score_predictions(truth, predicted) for predicted in (before, after)]\n        for name, metrics in zip(("baseline_ce", "utterance_evidence"), scores):\n            if metrics != by_name[name]["metrics"]:\n                raise ValueError("Candidate comparison metrics disagree with saved predictions")\n        report.update(\n            samples=len(truth), baseline_macro_f1=scores[0]["macro_f1"],\n            utterance_evidence_macro_f1=scores[1]["macro_f1"],\n            macro_f1_delta=scores[1]["macro_f1"] - scores[0]["macro_f1"],\n            per_symptom={},\n        )\n        for index, label in enumerate(data.TARGET_SYMPTOMS):\n            old_correct, new_correct = before[:, index] == truth[:, index], after[:, index] == truth[:, index]\n            old, new = (metrics["per_symptom"][label] for metrics in scores)\n            report["per_symptom"][label] = {\n                "baseline": old, "utterance_evidence": new,\n                "f1_delta": new["f1"] - old["f1"],\n                "corrected_errors": int((~old_correct & new_correct).sum()),\n                "new_errors": int((old_correct & ~new_correct).sum()),\n                "persistent_errors": int((~old_correct & ~new_correct).sum()),\n                "positive_predictions_before": int(before[:, index].sum()),\n                "positive_predictions_after": int(after[:, index].sum()),\n            }\n    else:\n        report["reason"] = "Only one candidate completed; no paired method comparison is available"\n    data.write_json(root / "candidate_comparison.json", report)\n\n\ndef _final_model(selected, texts, targets, holdout, settings, root, budget):\n    checkpoint, ready = root / "mission3_ver18_model.ckpt", root / "final_model_ready.json"\n    if ready.is_file():\n        saved = _read(ready)\n        if saved["selected_candidate"] != selected["candidate_id"]:\n            raise ValueError("Final model selection changed during resume")\n        _verify_hashes(root, saved["artifact_hashes"], budget)\n        return checkpoint, _read(checkpoint)\n    manifest = dict(selected["manifest"])\n    reason = "budget_before_refit"\n    with np.load(_safe(root, selected["directory"] + "/holdout_predictions.npz"), allow_pickle=False) as saved:\n        expected_probe = saved["predictions"][:12].copy()\n    probe_texts = [texts[i] for i in holdout[:len(expected_probe)]]\n    estimate = _refit_estimate(selected, len(targets) / manifest["fitted_samples"])\n    if budget.can_start(estimate, training=True):\n        attempt = _stamp()\n        local = Path(settings["work_dir"]) / "training" / root.name / "full_refit" / attempt\n        local.mkdir(parents=True, exist_ok=False)\n        final_settings = {**settings, **selected["settings"]}\n        data.write_json(local / "settings.json", _stable_settings(final_settings))\n        try:\n            new_manifest, new_probe = refit_selected(texts, targets, final_settings,\n                                                     manifest["selected_epochs"], local, budget)\n        except BudgetExceeded:\n            reason = "budget_during_refit"\n            print("Refit did not finish within its training budget; retaining the completed fit-only candidate.", flush=True)\n        else:\n            destination = root / "final_model" / attempt\n            _publish_tree(local, destination, budget)\n            relative = destination.relative_to(root).as_posix()\n            manifest = {**new_manifest, "weights": relative + "/" + new_manifest["weights"],\n                        "assets": relative + "/" + new_manifest["assets"]}\n            expected_probe, probe_texts = new_probe, texts[:len(new_probe)]\n            data.write_json(root / "final_history.json", _read(local / "final_history.json"))\n            reason = None\n    else:\n        print("Full refit does not fit the remaining budget; using the completed fit-only model.", flush=True)\n    manifest.update(selected_candidate=selected["candidate_id"], refit_skipped_reason=reason)\n    data.write_json(checkpoint, manifest)\n    actual = predict_checkpoint(checkpoint, probe_texts, budget)\n    if not np.array_equal(actual, expected_probe):\n        raise ValueError("Reloaded final model differs from its saved selection or in-memory predictions")\n    hashes = _model_hashes(root, manifest, (checkpoint.name,))\n    budget.check("commit verified final model")\n    data.write_json(ready, {"selected_candidate": selected["candidate_id"], "reloaded_probe_samples": len(expected_probe),\n                           "artifact_hashes": hashes})\n    return checkpoint, manifest\n\n\ndef _environment():\n    import torch\n    from importlib.metadata import version, PackageNotFoundError\n    packages = {}\n    for name in ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification", "tokenizers"):\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            packages[name] = "not installed"\n    return {"packages": packages, "cuda_available": torch.cuda.is_available(),\n            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}\n\n\ndef run(settings, runtime_sources=None):\n    started = time.monotonic()\n    settings = validate_settings(settings)\n    deadline = settings.get("deadline_monotonic", started + settings["time_budget_minutes"] * 60)\n    if deadline > started + settings["time_budget_minutes"] * 60 + 1:\n        raise ValueError("Supplied deadline exceeds the allowed execution budget")\n    budget = Budget(deadline, finalization_reserve_s=settings["finalization_reserve_seconds"],\n                    safety_factor=settings["budget_safety_factor"])\n    sources = runtime_sources or {p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.iterdir()\n                                 if p.is_file() and p.suffix in (".py", ".md", ".txt")}\n    sources = {key: value for key, value in sources.items() if not key.startswith("test_") and key != "build_notebook.py"}\n    output = Path(settings["output_dir"]).expanduser().resolve()\n    resume = settings.get("resume_run", "")\n    root = Path(resume).expanduser().resolve() if resume else output / _stamp()\n    if root == output or not root.is_relative_to(output):\n        raise ValueError("resume_run must be a child run directory inside output_dir")\n    if resume and not (root / "run_identity.json").is_file():\n        raise ValueError("resume_run is not initialized")\n    if not resume:\n        root.mkdir(parents=True, exist_ok=False)\n    print("Result run:", root, flush=True)\n    writable_status = not bool(resume)\n    records, skipped = [], []\n    try:\n        budget.check("prepare inputs")\n        _device(settings)\n        train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n                                                settings.get("train_dir", ""), settings.get("validation_dir", ""))\n        paths, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n        identity = {"version": "ver18", "training_digest": dataset_digest(texts, targets),\n                    "source_digest": _digest({k: v for k, v in sources.items() if not k.endswith(".md")}),\n                    "settings_digest": _digest(_stable_settings(settings)), "candidates": CANDIDATES,\n                    "model_selection": "Training internal holdout Macro F1 only"}\n        # JSON normalizes tuples, so identity is stored and compared in that canonical representation.\n        identity = json.loads(json.dumps(identity))\n        if resume:\n            if _read(root / "run_identity.json") != identity:\n                raise ValueError("Resume source, settings, or Training data changed")\n            with np.load(root / "training_split.npz", allow_pickle=False) as saved:\n                if not np.array_equal(saved["fit"], fit) or not np.array_equal(saved["holdout"], holdout):\n                    raise ValueError("Resume Training split changed")\n        else:\n            data.write_json(root / "run_identity.json", identity)\n            data.write_json(root / "settings.json", {k: v for k, v in settings.items() if k != "deadline_monotonic"})\n            np.savez_compressed(root / "training_split.npz", fit=fit, holdout=holdout)\n            submission = root / "submission"\n            submission.mkdir()\n            for name, source in sources.items():\n                if Path(name).name != name:\n                    raise ValueError("Runtime sources must have flat filenames")\n                (submission / name).write_text(source, encoding="utf-8")\n        if (root / "run_complete.json").is_file():\n            _verify_hashes(root, _read(root / "run_complete.json")["artifact_hashes"], budget)\n            print("Run already complete; its saved final artifacts were verified.", flush=True)\n            return root\n        writable_status = True\n        data.write_json(root / "runtime.json", {"status": "running", "seconds": time.monotonic() - started,\n                        "message": "Only run_complete.json confirms completed final artifacts."})\n        data.write_json(root / ("environment_" + _stamp() + ".json"), _environment())\n        data.write_json(root / "experiment_plan.json", {"candidates": CANDIDATES,\n                        "positive_class_weights": POSITIVE_CLASS_WEIGHTS, "ce_views": 1,\n                        "selection": "Training Macro F1 only", "max_epochs": settings["max_epochs"],\n                        "scheduler_horizon_epochs": settings["scheduler_horizon_epochs"],\n                        "time_budget_minutes": settings["time_budget_minutes"],\n                        "finalization_reserve_seconds": settings["finalization_reserve_seconds"],\n                        "budget_fallback": "completed fit-only candidate; never incomplete refit"})\n        final_already_ready = (root / "final_model_ready.json").is_file()\n        for candidate in CANDIDATES:\n            name = candidate["candidate_id"]\n            completed = root / "experiments" / name / "completed.json"\n            if not completed.is_file() and final_already_ready:\n                skipped.append({"candidate": name, "reason": "final_model_already_selected"})\n                continue\n            if not completed.is_file() and records:\n                previous = records[0]\n                minimum_trial = max(row["epoch_seconds"] for row in previous["history"]) * min(3, settings["max_epochs"]) + 120\n                current_best = max(records, key=lambda row: row["metrics"]["macro_f1"])\n                estimate = minimum_trial + _refit_estimate(current_best, len(targets) / len(fit))\n                if not budget.can_start(estimate, training=True):\n                    skipped.append({"candidate": name, "reason": "budget_reserving_refit"})\n                    print("Skipping additional candidate to preserve finalization/refit time:", name, flush=True)\n                    continue\n            config = {**settings, **candidate, "positive_class_weights": list(POSITIVE_CLASS_WEIGHTS)}\n            try:\n                records.append(_candidate_record(config, texts, targets, fit, holdout, root, budget))\n            except BudgetExceeded:\n                skipped.append({"candidate": name, "reason": "budget_no_completed_epoch"})\n                if not records:\n                    raise\n                break\n            _write_table(root, records)\n        if not records:\n            raise BudgetExceeded("No completed candidate is available")\n        selected = max(records, key=lambda row: row["metrics"]["macro_f1"])\n        _write_comparison(root, records, selected, budget)\n        data.write_json(root / "final_selection.json", {"candidate_id": selected["candidate_id"],\n                        "metrics": selected["metrics"], "selection_policy": "Training Macro F1 only",\n                        "selected_epochs": selected["manifest"]["selected_epochs"], "skipped_candidates": skipped})\n        data.write_json(root / "training_metrics.json", selected["metrics"])\n        saved = _safe(root, selected["directory"] + "/holdout_predictions.npz")\n        shutil.copy2(saved, root / "holdout_predictions.npz")\n        with np.load(saved, allow_pickle=False) as values:\n            data.export_errors([paths[i] for i in holdout], [texts[i] for i in holdout], targets[holdout],\n                               values["predictions"], root / "training_errors")\n        checkpoint, manifest = _final_model(selected, texts, targets, holdout, settings, root, budget)\n        hashes = _read(root / "final_model_ready.json")["artifact_hashes"]\n        hashes["candidate_comparison.json"] = _hash(root / "candidate_comparison.json")\n        if settings.get("run_validation", True):\n            predicted_paths, predicted = inference(checkpoint, validation_dir, root / "mission3.csv", budget)\n            budget.check("score final Validation CSV")\n            scored_paths, _, truth = data.load_dataset(validation_dir, include_targets=True)\n            if [p.name for p in scored_paths] != [p.name for p in predicted_paths]:\n                raise ValueError("Validation row order changed")\n            metrics = data.score_predictions(truth, predicted)\n            data.write_json(root / "validation_metrics.json", metrics)\n            print(f"Final Validation Macro F1={metrics[\'macro_f1\']:.6f}", flush=True)\n            for name in ("mission3.csv", "validation_metrics.json"):\n                hashes[name] = _hash(root / name)\n        limited = bool(skipped) or manifest["training_scope"] != "all_training" or any(r["stop_reason"].startswith("budget") for r in records)\n        budget.check("commit final run")\n        data.write_json(root / "runtime.json", {"status": "complete_budget_limited" if limited else "complete",\n                        "seconds": time.monotonic() - started, "seconds_scope": "worker invocation; notebook deadline includes installation",\n                        "candidate_count": len(records), "selected_candidate": selected["candidate_id"],\n                        "training_scope": manifest["training_scope"], "fitted_samples": manifest["fitted_samples"],\n                        "refit_skipped_reason": manifest["refit_skipped_reason"], "skipped_candidates": skipped,\n                        "budget_minutes_remaining": max(0., budget.remaining() / 60)})\n        data.write_json(root / "run_complete.json", {"artifact_hashes": hashes})\n    except BudgetExceeded:\n        if writable_status:\n            data.write_json(root / "runtime.json", {"status": "budget_exhausted", "seconds": time.monotonic() - started,\n                            "candidate_count": len(records), "skipped_candidates": skipped,\n                            "message": "No newly completed run; existing completed candidate artifacts are preserved."})\n        print("Time budget exhausted. Completed artifacts remain in the result directory.", flush=True)\n    except BaseException as error:\n        if writable_status:\n            data.write_json(root / "runtime.json", {"status": "interrupted" if isinstance(error, (KeyboardInterrupt, SystemExit)) else "failed",\n                            "seconds": time.monotonic() - started, "error_type": type(error).__name__})\n        raise\n    return root\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--audio_dir", required=True)\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args(argv)\n    inference(args.ckpt_path, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""Official one-command inference entry point for a portable result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_categorical_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Launch the bounded ver18 experiment in a fresh interpreter after installation."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\n\nSOURCE_FILES = (\n    "__init__.py", "data.py", "bootstrap.py", "neural.py", "budget.py", "runtime.py",\n    "inference.py", "run_notebook.py", "requirements.txt", "README.md",\n)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver18_execution"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    runtime = importlib.import_module(name + ".runtime")\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n    sources = {name: (root / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n    result = runtime.run(settings, runtime_sources=sources)\n    Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': "# Mission 3 ver18\n\n## Experiment: global context with utterance evidence\n\n- `baseline_ce`: the ver15 classifier with moderate positive-class weights.\n- `utterance_evidence`: the same classifier plus a small nausea evidence branch over ordered utterances.\n\nBoth start from `klue/roberta-base` revision `02f94ba5e3fcb7e2a58a390b8639b0fac974a8da`. The evidence branch reuses the same encoder outputs rather than running another encoder pass. It pools token representations into original utterances, joins previous/current/next utterance representations in order, and learns which contextualized utterances provide evidence. An utterance split across windows is reunited, with inverse occurrence weights reducing duplicated overlap contributions. Every utterance and the original global context branch are retained.\n\nThe small evidence branch has hidden width 128 and dropout 0.1. Its zero-initialized output residual updates only the two nausea logits during training. The encoder is jointly trained, so other labels can change through shared representations. Special tokens and padding do not become evidence. Empty documents use the global prediction with a zero evidence residual. The manifest stores the architecture for strict reconstruction.\n\nThis tests whether localized symptom expressions and surrounding questions/answers help nausea classification. It does not assume that attention automatically understands negation, that missing keywords mean incorrect labels, or that existing labels should be replaced. It differs from ver14 token-level label attention by aggregating original utterances with ordered neighbors. CorNet, TAPT, GCE, keyword filtering, and post-training score adjustments are not added.\n\n## Execution and safeguards\n\nOpen `mission3_ver18.ipynb` in Colab, select a GPU, and run all cells using the existing `mission3_ver5_inputs.zip`. Select the shared Drive account when prompted. All runtime sources are embedded; no earlier result is required. Results go to unique run directories under `DCC_AIdeas/results/mission3/ver18` in Drive.\n\nInputs remain `utterances[].text`. All original targets and samples remain unchanged. Utterance order and character offsets come from these text fields, not speaker or medical annotations. The nine two-class outputs use ordinary argmax, without thresholds, post-training calibration, external training corpora, or automatic relabeling. Both candidates use the same grouped Training split, seed, fixed KLUE revision, weighted cross entropy, and moderate positive weights `[1.4, 1.8, 2.0, 1.2, 1.3, 1.0, 3.0, 1.5, 1.3]`. Candidate training uses at most three epochs with the original four-epoch learning-rate schedule horizon. GPU execution is not guaranteed bitwise reproducible across runs.\n\nTraining holdout Macro F1 selects the candidate and epoch. Official Validation is evaluated only after final selection. A selected full refit starts fresh and uses all Training targets for the selected number of epochs. If the new branch is worse on the internal selection metric, the baseline can win; both candidates' internal results remain available.\n\nThe shared 180-minute deadline starts after Drive login, before installation. Installation also has a 20-minute cap. The runtime reserves 20 minutes for finalization and uses measured time with a 1.25 safety factor. Slow hardware may require skipping a candidate or full refit. Only complete epochs and fully complete refits are published; incomplete refits never replace completed candidates. A fit-only fallback records its actual `training_scope`, `fitted_samples`, and `refit_skipped_reason`. A deadline cannot guarantee all phases finish on every GPU.\n\n`AUTO_RELEASE_GPU=True` requests Colab runtime release after result display, errors, or deadline. Drive results survive; `/content` scratch may disappear. If release fails, use Runtime > Disconnect and delete runtime manually. Actual Colab release service behavior and new-method GPU duration remain unverified locally.\n\n## Saved results and resuming\n\nKeep the entire result directory and executed notebook. Important files are `candidate_summary.csv`, `candidate_comparison.json`, `final_selection.json`, `training_metrics.json`, `validation_metrics.json`, `runtime.json`, `mission3.csv`, and `mission3_ver18_model.ckpt` with referenced weights/assets. The portable inference entry point is `submission/inference.py`.\n\n`candidate_comparison.json` compares the two candidates on the same Training holdout: per-label precision/recall/F1 changes and paired counts of corrected, newly introduced, and persistent errors. It contains no dialogue text or sample identifiers. This diagnostic does not change the selection metric or tune on official Validation. If a candidate cannot complete within budget, the report records that comparison is unavailable.\n\nOnly `complete` or `complete_budget_limited` status plus `run_complete.json` represents completed final work. Missing final files or completion marker, `budget_exhausted`, `failed`, or `interrupted` mean incomplete work. Abrupt Colab reclamation can leave the previous saved status unchanged.\n\nSet `resume_run` to a compatible run directory. Source, settings, data, split, and artifact hashes are checked before completed artifacts are reused. Partial epochs are not resumed. Resuming an already completed run only verifies and returns it without extending training. Each explicit rerun consumes a new budget and additional resources.\n\nPrevious versions and dependency requirements are unchanged. New full-data F1 improvement is unverified until the notebook is run.\n"}
try:
    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver18_"))
    for name, source in RUNTIME_SOURCES.items():
        (runtime_dir / name).write_text(source, encoding="utf-8")
    budget_spec = importlib.util.spec_from_file_location("mission3_ver18_budget_supervisor", runtime_dir / "budget.py")
    budget_module = importlib.util.module_from_spec(budget_spec)
    sys.modules[budget_spec.name] = budget_module
    budget_spec.loader.exec_module(budget_module)
    print("실행 코드 준비 완료")
except BaseException:
    release_gpu()
    raise


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비 (최대 20분, 전체 예산에 포함)
package_deadline = min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60)
try:
    budget_module.run_supervised(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],
        deadline_monotonic=package_deadline, shutdown_margin_s=10,
    )
    print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")
except BaseException:
    release_gpu()
    raise


In [ ]:
# @title 후보 학습 → 최종 평가 · 결과 표시 · GPU 반환
import csv
import json
try:
    settings_file = runtime_dir / "notebook_settings.json"
    result_file = runtime_dir / "notebook_result.json"
    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    # A fresh interpreter uses installed versions; the deadline includes all work.
    budget_module.run_supervised(
        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
         "--settings", str(settings_file), "--result_file", str(result_file)],
        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,
    )
    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))
    print("결과 폴더:", result_dir)
    summary = result_dir / "candidate_summary.csv"
    if summary.is_file():
        print("Training 내부 비교:")
        with summary.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                      "오심 F1 =", round(float(row["nausea_f1"]), 6),
                      "선택 epoch =", row["selected_epochs"], "종료 이유 =", row["stop_reason"])
    comparison_path = result_dir / "candidate_comparison.json"
    if comparison_path.is_file():
        comparison = json.loads(comparison_path.read_text(encoding="utf-8-sig"))
        if comparison["comparison_available"]:
            nausea = comparison["per_symptom"]["오심"]
            print("내부 평가 오심 변화: F1", round(nausea["f1_delta"], 6),
                  "· 고친 오답", nausea["corrected_errors"],
                  "· 새 오답", nausea["new_errors"],
                  "· 두 후보 공통 오답", nausea["persistent_errors"])
        else:
            print("두 후보 비교 미완료:", comparison["reason"])
    for filename in ["training_metrics.json", "validation_metrics.json"]:
        path = result_dir / filename
        if path.is_file():
            report = json.loads(path.read_text(encoding="utf-8-sig"))
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
            for symptom, values in report["per_symptom"].items():
                print(symptom, "F1 =", round(values["f1"], 4),
                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))
    manifest_path = result_dir / "mission3_ver18_model.ckpt"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        print("선택된 후보:", manifest.get("selected_candidate", "모델 정보 확인"))
        print("저장 모델 학습 범위:", manifest["training_scope"],
              "· 학습 표본:", manifest["fitted_samples"], "· epoch:", manifest["selected_epochs"])
        if manifest.get("refit_skipped_reason"):
            print("전체 재학습 생략 이유:", manifest["refit_skipped_reason"])
    status_path = result_dir / "runtime.json"
    status = {}
    if status_path.is_file():
        status = json.loads(status_path.read_text(encoding="utf-8-sig"))
        print("실행 상태:", status.get("status", "기록 확인 필요"))
        if status.get("skipped_candidates"):
            print("생략한 후보:", status["skipped_candidates"])
    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")
    print("선택 근거:", result_dir / "final_selection.json")
    print("학습 범위가 기록된 모델 정보:", manifest_path)
    complete = ((result_dir / "run_complete.json").is_file()
                and status.get("status") in {"complete", "complete_budget_limited"})
    if complete:
        print("최종 작업 완료 · 예측:", result_dir / "mission3.csv")
    else:
        print("미완료 실행:", status.get("status", "상태 기록 없음"),
              "· 저장된 후보 결과와 로그는 결과 폴더에 남아 있습니다. 최종 제출 결과로 사용하지 마세요.")
finally:
    release_gpu()


## 결과 확인 · 중단 후 재개

기본 실행은 `ver18/실행시각_고유번호/`에 저장됩니다. 이전 버전의 노트북과 결과는 유지합니다.
`candidate_summary.csv`, `candidate_comparison.json`, `final_selection.json`, `training_metrics.json`,
`validation_metrics.json`, `runtime.json`을 확인하세요.
모델 `.ckpt`의 `training_scope`가 전체 Training 학습인지 내부 fit 학습인지를 알려 줍니다.
시간 때문에 전체 재학습을 생략한 경우에도 실제 완료 범위를 명시합니다.
`budget_exhausted` 또는 최종 예측·평가 파일 누락은 완료된 제출 결과가 아닙니다.
강제 종료·Colab 회수 시 마지막 `runtime.json` 상태가 최종 종료 이유를 반영하지 못할 수 있습니다.
`run_complete.json`은 최종 완료 표시이므로 이 파일이 없는 실행을 완료로 간주하지 마세요.

재개하려면 `resume_run`에 이전 실행 폴더의 전체 Drive 경로를 넣고 다시 실행합니다.
재개 가능 여부는 코드·설정·데이터 일치 여부로 확인하며, 완료된 결과만 재사용합니다.
이미 `complete` 또는 `complete_budget_limited`인 실행을 재개하면 저장된 결과를 검증하고 반환하며,
추가 학습하지 않습니다. 새 실험을 하려면 `resume_run`을 비우세요.
중단된 epoch 중간부터 이어 학습하지 않습니다. **재실행에는 새 시간 예산과 추가 Colab 리소스가 듭니다.**

모델 `.ckpt`는 가중치·assets의 상대 경로를 담은 manifest입니다.
**실행 폴더 전체와 실행 로그가 있는 노트북을 보관**하세요. 제출용 추론 코드는 `submission/inference.py`입니다.
예측 CSV의 헤더는 `label file name,symptom`입니다.
기본 설정은 결과 표시 후 또는 제한 시간에 Colab `runtime.unassign()`으로 GPU 반환을 요청합니다.
`AUTO_RELEASE_GPU`를 끄거나 자동 반환이 실패하면 **런타임 → 연결 해제 및 런타임 삭제**를 직접 실행하세요.
자동 반환 기능의 실제 서비스 동작은 Colab에서 확인해야 합니다.
